### Importing packages

In [351]:
import pandas as pd                                                                     # type: ignore

### Importing dataset

This is only data for one-hour lead-time

In [352]:
test_data_X = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/test-data-Dakar-map-features.csv")
test_data_y_lt1 = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/test-data-Dakar-map-target-lt1.csv", index_col=False)

train_data_X = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/train-data-Dakar-map-features.csv", index_col=False)
train_data_y_lt1 = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/train-data-Dakar-map-target-lt1.csv", index_col=False)

### Choosing lead time

In [353]:
lead_time = 3

### Choosing data split

In [354]:
dataset = "train"
if dataset == "train":
    data = train_data_X
    target = train_data_y_lt1
else:
    data = test_data_X
    target = test_data_y_lt1

In [355]:
original_data = data.copy()

### Finding exact rows after given time (hh, mm)

In [356]:
data['datetime'] = pd.to_datetime(data[['year', 'month', 'day', 'hour', 'minute']])
data = data.sort_values(by='datetime').reset_index(drop=True)

In [357]:
def find_exact_row_after_given_hours(row, hours, minutes, df):
    target_time = row['datetime'] + pd.Timedelta(hours=hours, minutes=minutes)
    corresponding_row = df[df['datetime'] == target_time]
    if not corresponding_row.empty:
        return corresponding_row.index[0]  # Return the index of the corresponding row
    else:
        return None

In [358]:
data['row_index_X0_15'] = data.apply(find_exact_row_after_given_hours, args=(0, 15, data), axis=1)
data['row_index_X0_30'] = data.apply(find_exact_row_after_given_hours, args=(0, 30, data), axis=1)
data['row_index_X0_45'] = data.apply(find_exact_row_after_given_hours, args=(0, 45, data), axis=1)
data['row_index_X0_60'] = data.apply(find_exact_row_after_given_hours, args=(1, 0, data), axis=1)

### Find the target index for the chosen lead time for each row

We need $t_0$, $t_1$ and $t_{lt}$

Target $C_b$ is one hour after the observation time ($t_0$)

In [359]:
data['row_index_Cb'] = data.apply(find_exact_row_after_given_hours, args=(lead_time, 0, data), axis=1)      # since the target is at t0+1 h

In [360]:
columns_to_keep = original_data.keys().to_list()

In [361]:
# Function to combine current row with rows based on indices, retaining original column order
def combine_current_and_rows(row, data):
    # Extract the row indices from the current row
    indices = [row['row_index_X0_15'], row['row_index_X0_30'], row['row_index_X0_45'], row['row_index_X0_60']]
    
    # Fetch the current row (filter only relevant columns)
    current_row = row[columns_to_keep].copy()
    
    # List to store the rows
    rows_to_combine = [current_row]
    
    # Fetch rows corresponding to the indices, rename columns with suffix to avoid duplicates
    for i, idx in enumerate(indices):
        if pd.notna(idx):
            # Fetch the row, keep only relevant columns, and rename them with suffix
            fetched_row = data.loc[idx, columns_to_keep].rename(lambda col: f"{col}_{(i+1)*15}")
            rows_to_combine.append(fetched_row)
        else:
            # If the index is NaN, create an empty Series with the same columns as the current row
            empty_row = pd.Series(index=[f"{col}_{(i+1)*15}" for col in columns_to_keep])
            rows_to_combine.append(empty_row)
    
    # Concatenate the current row with the fetched rows side by side (maintain column order)
    combined_row = pd.concat(rows_to_combine, axis=0)
    
    return combined_row
# Apply the function row by row to get combined data
combined_data = data.apply(combine_current_and_rows, args=(data,), axis=1)

# Convert the combined series into a DataFrame while keeping the original index
combined_df = pd.DataFrame(combined_data, index=data.index)

### Taking target values

In [362]:
combined_df = pd.DataFrame(combined_data, index=data.index)

In [364]:
combined_df = combined_df.dropna()

In [365]:
combined_df

,year,month,day,hour,minute,lat1,lon1,lat2,lon2,lat3,...,size1_60,size2_60,size3_60,size4_60,size5_60,ds1_60,ds2_60,ds3_60,ds4_60,ds5_60
0,2008,6,1,0,0,15.18,-13.12,15.81,-12.09,14.06,...,5550.0,4425.0,0.0,0.0,0.0,101.65,129.25,400.00,400.00,400.00
1,2008,6,1,0,15,15.27,-13.07,15.00,-11.95,14.00,...,6025.0,2800.0,0.0,0.0,0.0,100.96,130.25,400.00,400.00,400.00
2,2008,6,1,0,30,16.04,-12.40,15.05,-12.22,15.32,...,3250.0,2875.0,1375.0,2250.0,100.0,98.08,106.21,133.18,147.80,166.33
3,2008,6,1,0,45,15.09,-13.57,14.73,-12.71,14.00,...,2825.0,2675.0,975.0,250.0,650.0,98.08,109.33,129.84,133.06,142.37
4,2008,6,1,1,0,14.69,-12.71,15.23,-13.66,14.00,...,2425.0,1600.0,1850.0,1425.0,0.0,98.08,109.33,132.77,160.51,400.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69084,2019,9,30,3,30,12.22,-16.13,10.87,-16.22,13.21,...,4150.0,6175.0,15325.0,14625.0,0.0,50.54,70.71,95.57,116.21,400.00
69085,2019,9,30,3,45,10.87,-16.26,12.13,-15.32,12.89,...,3650.0,4125.0,1875.0,14925.0,0.0,48.47,68.73,95.34,100.40,400.00
69086,2019,9,30,4,0,12.26,-16.13,13.39,-14.20,13.03,...,1450.0,2000.0,2300.0,14650.0,0.0,46.65,65.76,88.32,100.40,400.00
69087,2019,9,30,4,15,13.12,-15.72,11.14,-16.44,14.00,...,1100.0,1325.0,5225.0,11800.0,11800.0,45.28,61.98,86.98,97.25,113.00


In [366]:
o = data['row_index_Cb'][combined_df.index].dropna()

In [367]:
combined_df.loc[o.index]

,year,month,day,hour,minute,lat1,lon1,lat2,lon2,lat3,...,size1_60,size2_60,size3_60,size4_60,size5_60,ds1_60,ds2_60,ds3_60,ds4_60,ds5_60
0,2008,6,1,0,0,15.18,-13.12,15.81,-12.09,14.06,...,5550.0,4425.0,0.0,0.0,0.0,101.65,129.25,400.00,400.00,400.00
1,2008,6,1,0,15,15.27,-13.07,15.00,-11.95,14.00,...,6025.0,2800.0,0.0,0.0,0.0,100.96,130.25,400.00,400.00,400.00
2,2008,6,1,0,30,16.04,-12.40,15.05,-12.22,15.32,...,3250.0,2875.0,1375.0,2250.0,100.0,98.08,106.21,133.18,147.80,166.33
3,2008,6,1,0,45,15.09,-13.57,14.73,-12.71,14.00,...,2825.0,2675.0,975.0,250.0,650.0,98.08,109.33,129.84,133.06,142.37
4,2008,6,1,1,0,14.69,-12.71,15.23,-13.66,14.00,...,2425.0,1600.0,1850.0,1425.0,0.0,98.08,109.33,132.77,160.51,400.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69078,2019,9,30,2,0,12.76,-14.06,12.08,-14.33,11.55,...,9450.0,11550.0,0.0,0.0,0.0,82.10,102.96,400.00,400.00,400.00
69079,2019,9,30,2,15,11.95,-15.32,12.85,-14.06,12.67,...,9850.0,13900.0,0.0,0.0,0.0,87.13,103.39,400.00,400.00,400.00
69080,2019,9,30,2,30,12.94,-14.38,12.76,-13.39,11.95,...,2100.0,6825.0,15550.0,600.0,0.0,80.22,86.95,105.12,128.00,400.00
69082,2019,9,30,3,0,13.16,-14.38,11.59,-15.86,14.00,...,3175.0,24125.0,1650.0,700.0,0.0,56.80,78.23,84.08,87.09,400.00


In [368]:
target

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69090,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
69091,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
69092,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
69093,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [369]:
target_index = data['row_index_Cb'][combined_df.index].dropna()
target = target.loc[target_index]
combined_df = combined_df.loc[target_index.index]

### Writing data to disk

In [370]:
combined_df.to_csv(f"{dataset}-Dakar-map-features-many-timesteps-lt{lead_time}.csv", index=False)
target.to_csv(f"{dataset}-Dakar-map-target--many-timesteps.lt{lead_time}.csv", index=False)